In [ ]:
import os
import sys
import torch
import dill
import numpy as np
import collections
import imageio
import cv2  
from scipy.spatial.transform import Rotation as R
from scipy.spatial import ConvexHull


import robomimic.utils.file_utils as FileUtils
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils

##############################################################################
# 1) SAFE SET LOADING & CHECKING
##############################################################################

def load_safe_set(file_path="/Riad/vivid123/safe_set_6d_simulation.npz"):
    data = np.load(file_path)
    safe_set = data["safe_set"]              # shape (N,6)
    hull_equations = data["hull_equations"]  # shape (m,7)
    hull_vertices = data["hull_vertices"]
    return safe_set, hull_equations, hull_vertices

def normalize_quaternion(q):
    norm = np.linalg.norm(q)
    if norm < 1e-6:
        return np.array([0,0,0,1])
    return q / norm

def pose7d_to_6d(pose7d):
    """
    Convert [x, y, z, qx, qy, qz, qw] -> [x, y, z, rx, ry, rz]
    where rx, ry, rz is the rotation vector from the quaternion.
    """
    pos = pose7d[:3]
    quat = normalize_quaternion(pose7d[3:7])
    rotvec = R.from_quat(quat).as_rotvec()
    return np.hstack([pos, rotvec])

def is_pose_in_safe_set_6d(query_6d, hull_equations, tol=1e-1):
    """
    Check if a 6D pose is inside the convex hull:
       A.dot(query_6d) + b <= tol
    for all facets.

    NOTE: Using tol=1e-1 as specified.
    """
    A = hull_equations[:, :-1]  # shape (m,6)
    b = hull_equations[:, -1]   # shape (m,)
    distances = np.dot(A, query_6d) + b
    inside = np.all(distances <= tol)
    return inside, distances

##############################################################################
# 2) APPLY A 6D DELTA (POS + EULER) IN THE LOCAL FRAME
##############################################################################

def apply_local_pose_delta(current_pose_7d, delta_6d,
                           max_pos=0.05, max_ori=0.5):
    """
    current_pose_7d: [x, y, z, qx, qy, qz, qw]  (absolute)
    delta_6d: [dx, dy, dz, droll, dpitch, dyaw] in [-1,1]
      - We scale them by output_max = [0.05, 0.05, 0.05, 0.5, 0.5, 0.5]
      - Then apply in the local frame.
    Returns:
      new_pose_7d: predicted absolute pose after applying local delta
    """
    dx = delta_6d[0] * max_pos
    dy = delta_6d[1] * max_pos
    dz = delta_6d[2] * max_pos
    droll = delta_6d[3] * max_ori
    dpitch = delta_6d[4] * max_ori
    dyaw = delta_6d[5] * max_ori

    old_pos = current_pose_7d[:3]
    old_quat = normalize_quaternion(current_pose_7d[3:7])
    old_rot = R.from_quat(old_quat)

    # Orientation update
    delta_rot = R.from_euler('xyz', [droll, dpitch, dyaw], degrees=False)
    new_rot = old_rot * delta_rot
    new_quat = new_rot.as_quat()

    # Position update in local frame
    local_trans = np.array([dx, dy, dz])
    local_trans_world = old_rot.apply(local_trans)
    new_pos = old_pos + local_trans_world

    new_pose_7d = np.concatenate([new_pos, new_quat])
    return new_pose_7d

##############################################################################
# 3) FRAME STACKER & DUMMY WRAPPER
##############################################################################

class FrameStackForTrans:
    def __init__(self, num_frames):
        self.num_frames = num_frames
        self.obs_history = {}
    
    def reset(self, init_obs):
        self.obs_history = {}
        for k in init_obs:
            self.obs_history[k] = collections.deque(
                [init_obs[k][None] for _ in range(self.num_frames)],
                maxlen=self.num_frames
            )
        return {k: np.concatenate(self.obs_history[k], axis=0) for k in self.obs_history}
    
    def add_new_obs(self, new_obs):
        for k in new_obs:
            if 'timesteps' in k or 'actions' in k:
                continue
            self.obs_history[k].append(new_obs[k][None])
        return {k: np.concatenate(self.obs_history[k], axis=0) for k in self.obs_history}

class DummyObsWrapper:
    """
    If the env doesn't provide 'robot0_eye_in_hand_image',
    we inject a dummy key.
    """
    def __init__(self, env):
        self.env = env
        self.required_key = 'robot0_eye_in_hand_image'
        self.image_shape = (3, 84, 84)
    
    def reset(self):
        obs = self.env.reset()
        if self.required_key not in obs:
            obs[self.required_key] = np.zeros(self.image_shape, dtype=np.float32)
        return obs
    
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        if self.required_key not in obs:
            obs[self.required_key] = np.zeros(self.image_shape, dtype=np.float32)
        return obs, reward, done, info
    
    def render(self, *args, **kwargs):
        return self.env.render(*args, **kwargs)
    
    def __getattr__(self, name):
        return getattr(self.env, name)

def create_env(env_meta, shape_meta, enable_render=True):
    import robomimic.utils.obs_utils as ObsUtils
    import robomimic.utils.env_utils as EnvUtils
    modality_mapping = collections.defaultdict(list)
    for key, attr in shape_meta['obs'].items():
        modality_mapping[attr.get('type', 'low_dim')].append(key)
    ObsUtils.initialize_obs_modality_mapping_from_dict(modality_mapping)
    env = EnvUtils.create_env_from_metadata(
        env_meta=env_meta,
        render=False,
        render_offscreen=enable_render,
        use_image_obs=enable_render,
    )
    return env

import numpy as np
import cv2

def fix_image_for_opencv(img):
    """
    Ensures that 'img' is suitable for cv2.putText:
      - channel-last format: (H, W, 3) or (H, W, 4)
      - dtype = uint8
      - contiguous in memory
    """
    # Convert to numpy array if not already
    img = np.asanyarray(img)

    # 1) Check shape. If shape is (C,H,W), transpose to (H,W,C).
    if img.ndim == 3 and img.shape[0] in [1,3,4] and img.shape[1] > 10 and img.shape[2] > 10:
        img = np.transpose(img, (1,2,0))  # (C,H,W) -> (H,W,C)

    # 2) If the image is 2D (H,W), expand to 3D with 1 channel
    if img.ndim == 2:
        # convert grayscale to 3 channels
        img = np.stack([img]*3, axis=-1)


    # 4) Ensure dtype=uint8. If it's float, scale/clamp as needed.
    if img.dtype != np.uint8:
        # If max <= 1, we assume [0,1]. Otherwise, assume [0,255].
        maxval = img.max()
        if maxval <= 1.0:
            # scale up
            img = (img * 255.0).clip(0,255).astype(np.uint8)
        else:
            # clamp
            img = img.clip(0,255).astype(np.uint8)

    # 5) Make sure it's contiguous in memory
    img = np.ascontiguousarray(img)

    return img


##############################################################################
# 4) PUSH-UNTIL-OUT
##############################################################################

def rollout_push_until_out(env, hull_equations,
                           n_obs_steps=2, max_steps=200, return_imgs=False):
    """
    Single rollout:
      - If the robot is inside the safeset, we do random orientation-heavy pushes
        (one step at a time). After each push, we re-check if we're inside or outside.
      - Once we are out, we switch to normal policy for the rest of the rollout.
      - We do not break once out; we keep stepping until max_steps or done.

    We'll overlay text on each frame with inside/pushing info.
    """

    keys_select = ['robot0_eye_in_hand_image', 'agentview_image', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
    framestacker = FrameStackForTrans(n_obs_steps)
    obs = env.reset()
    obs = framestacker.reset(obs)

    step_count = 0
    pushing = True  # True => push mode, False => policy mode
    done = False
    imgs = []
    imgs_eye = []
    keep_push = 0
    trajectory = []

    while step_count < max_steps and not done:
        current_pose_7d = np.concatenate([
            obs['robot0_eef_pos'][-1],
            obs['robot0_eef_quat'][-1]
        ], axis=0)
        current_pose_6d = pose7d_to_6d(current_pose_7d)
        inside, dist = is_pose_in_safe_set_6d(current_pose_6d, hull_equations, tol=1e-1)

        print(f"Step {step_count}, inside safeset? {inside}, pushing? {pushing}")

        if pushing:
            if not inside:
                # we can keep pushing a few steps even after out
                if keep_push > 3:
                    print("We have pushed out of the safeset! Switching to policy mode.")
                    pushing = False
                keep_push += 1
            else:
                # random orientation push as example
                delta_6d = np.zeros(6, dtype=np.float32)
                delta_6d[:3] = np.random.uniform(-0.05, 0.05, size=3)
                delta_6d[3:] = np.random.uniform(-1, 1, size=3)

                final_action = np.concatenate([delta_6d, [0.0]], axis=0)
                next_obs, reward, done_env, info = env.step(final_action)
                step_count += 1

                # log trajectory
                try:
                    c_pose = np.concatenate([
                        next_obs['robot0_eef_pos'],
                        next_obs['robot0_eef_quat']
                    ], axis=0)
                    trajectory.append(c_pose[:3])
                except Exception as e:
                    print(e)

                if return_imgs:
                    img_agent = env.render(mode="rgb_array", height=512, width=512, camera_name="agentview")
                    img_eye = env.render(mode="rgb_array", height=512, width=512, camera_name="robot0_eye_in_hand")
                    img_agent = fix_image_for_opencv(img_agent)
                    img_eye   = fix_image_for_opencv(img_eye)

                    text_str = f"Inside: {inside}, pushing: {pushing}"
                    cv2.putText(img_agent, text_str, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                                1.0, (255, 255, 255), 2)
                    cv2.putText(img_eye, text_str, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                                1.0, (255, 255, 255), 2)

                    imgs.append(img_agent)
                    imgs_eye.append(img_eye)



                obs = framestacker.add_new_obs(next_obs)
                if done_env:
                    done = True
                if step_count >= max_steps or done:
                    break
                continue

        else:

            
            if return_imgs:
                img_agent = env.render(mode="rgb_array", height=512, width=512, camera_name="agentview")
                img_eye = env.render(mode="rgb_array", height=512, width=512, camera_name="robot0_eye_in_hand")

                img_agent = fix_image_for_opencv(img_agent)
                img_eye   = fix_image_for_opencv(img_eye)

                text_str = f"Inside: {inside}, pushing: {pushing}"
                cv2.putText(img_agent, text_str, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (255, 255, 255), 2)
                cv2.putText(img_eye, text_str, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (255, 255, 255), 2)

                imgs.append(img_agent)
                imgs_eye.append(img_eye)

                obs = framestacker.add_new_obs(next_obs)
                if done_env:
                    done = True

            break

    return imgs, imgs_eye, np.array(trajectory)

def undo_transform_action(action, rotation_transformer):
    raw_shape = action.shape
    if raw_shape[-1] == 20:
        action = action.reshape(-1, 2, 10)
    d_rot = action.shape[-1] - 4
    pos = action[..., :3]
    rot = action[..., 3:3+d_rot]
    gripper = action[..., -1:]
    rot = rotation_transformer.inverse(rot)
    uaction = np.concatenate([pos, rot, gripper], axis=-1)
    if raw_shape[-1] == 20:
        uaction = uaction.reshape(*raw_shape[:-1], 14)
    return uaction

# ----------------------------------------------------------------------
# 5) MAIN
# ----------------------------------------------------------------------
if __name__ == "__main__":

    # Load safe set
    safe_set, hull_equations, hull_vertices = load_safe_set("/Riad/vivid123/safe_set_6d_simulation.npz")

    dataset_path = "/Riad/diffusion_policy/full_image_low_lift_ph.hdf5"

    # create environment
    import robomimic.utils.file_utils as FileUtils
    env_meta = FileUtils.get_env_metadata_from_dataset(dataset_path)
    env_meta['env_kwargs']['controller_configs']['ramp_ratio'] = 1.0

    shape_meta = {
        'obs': {
            'robot0_eye_in_hand_image': {'shape': [3, 84, 84], 'type': 'rgb'},
            'agentview_image': {'shape': [3, 84, 84], 'type': 'rgb'},
            'robot0_eef_pos': {'shape': [3]},
            'robot0_eef_quat': {'shape': [4]},
            'robot0_gripper_qpos': {'shape': [2]},
        },
        'action': {
            'shape': [10]  # 6 for pose, 1 for gripper
        }
    }

    env = create_env(env_meta=env_meta, shape_meta=shape_meta, enable_render=True)
    env = DummyObsWrapper(env)

    # rotation_transformer = RotationTransformer('axis_angle', 'rotation_6d')

    # Do one rollout: push out, then run policy, with text overlays
    fps = 10
    imgs, imgs_eye, all_traj = rollout_push_until_out(
        env, hull_equations,
        n_obs_steps=2, max_steps=200,
        return_imgs=True
    )

    if imgs:
        imageio.mimwrite("push_agentview.mp4", imgs, fps=fps, quality=8)
        print("Saved push_agentview.mp4")
    if imgs_eye:
        imageio.mimwrite("push_eyeinhand.mp4", imgs_eye, fps=fps, quality=8)
        print("Saved push_eyeinhand.mp4")


Using device: cuda
Created environment with name Lift
Action size is 7
Step 0, inside safeset? True, pushing? True
Step 1, inside safeset? True, pushing? True
Step 2, inside safeset? True, pushing? True
Step 3, inside safeset? True, pushing? True
Step 4, inside safeset? True, pushing? True
Step 5, inside safeset? True, pushing? True
Step 6, inside safeset? True, pushing? True
Step 7, inside safeset? True, pushing? True
Step 8, inside safeset? True, pushing? True
Step 9, inside safeset? True, pushing? True
Step 10, inside safeset? True, pushing? True
Step 11, inside safeset? True, pushing? True
Step 12, inside safeset? True, pushing? True
Step 13, inside safeset? True, pushing? True
Step 14, inside safeset? True, pushing? True
Step 15, inside safeset? True, pushing? True
Step 16, inside safeset? True, pushing? True
Step 17, inside safeset? True, pushing? True
Step 18, inside safeset? True, pushing? True
Step 19, inside safeset? True, pushing? True
Step 20, inside safeset? True, pushing?

In [ ]:
import plotly.graph_objects as go
def visualize_trajectory(trajectory, safe_set_file="/Riad/vivid123/safe_set_6d_simulation.npz", tol=1e-1):
    """
    Visualize the rollout trajectory (positions) along with the safe set convex hull.
    """
    # Load safe set.
    data = np.load(safe_set_file)
    safe_set = data["safe_set"]
    safe_positions = safe_set[:, :3]
    # Build 3D convex hull on safe positions.
    from scipy.spatial import ConvexHull
    hull_3d = ConvexHull(safe_positions)
    
    fig = go.Figure()
    
    # Plot safe set hull.
    fig.add_trace(go.Mesh3d(
        x=safe_positions[:, 0],
        y=safe_positions[:, 1],
        z=safe_positions[:, 2],
        i=hull_3d.simplices[:, 0],
        j=hull_3d.simplices[:, 1],
        k=hull_3d.simplices[:, 2],
        opacity=0.3,
        color='lightblue',
        name='Safe Set Hull'
    ))
    
    # Plot trajectory.
    fig.add_trace(go.Scatter3d(
        x=trajectory[:, 0],
        y=trajectory[:, 1],
        z=trajectory[:, 2],
        mode='lines+markers',
        marker=dict(size=4, color='green'),
        line=dict(color='gray', width=2),
        name='Rollout Trajectory'
    ))
    
    fig.update_layout(
        title="Rollout Trajectory on Safe Set",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z")
    )
    fig.show()

# For visualization, we plot the trajectory from the first trial.

visualize_trajectory(all_traj, safe_set_file="/Riad/vivid123/safe_set_6d_simulation.npz")